# 03 — Feature Engineering & Selection
**Purpose:** Build engineered features, run per-target feature selection (RFECV + Boruta).

**Input:** `train_enriched.parquet`, `val_enriched.parquet` from notebook 01

**Output:** `train_featured.parquet`, `val_featured.parquet`, `config/feature_sets.yaml`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GroupKFold
import yaml
import warnings
warnings.filterwarnings('ignore')

SEED = 42
WORK_DIR = '/kaggle/working'

train = pd.read_parquet(f'{WORK_DIR}/train_enriched.parquet')
val = pd.read_parquet(f'{WORK_DIR}/val_enriched.parquet')
print(f'Train: {train.shape}, Val: {val.shape}')

## 1. Temporal Feature Engineering

In [ ]:
DATE_COL = 'Sample_Date'  # adjust

def add_temporal_features(df, date_col):
    """Add cyclical temporal features from date column."""
    df = df.copy()
    dt = pd.to_datetime(df[date_col])
    
    df['month'] = dt.dt.month
    df['quarter'] = dt.dt.quarter
    df['day_of_year'] = dt.dt.dayofyear
    df['year'] = dt.dt.year
    
    # Cyclical encoding (captures Dec→Jan continuity)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    
    # South Africa seasons (Southern Hemisphere)
    season_map = {12: 'summer', 1: 'summer', 2: 'summer',
                  3: 'autumn', 4: 'autumn', 5: 'autumn',
                  6: 'winter', 7: 'winter', 8: 'winter',
                  9: 'spring', 10: 'spring', 11: 'spring'}
    df['season'] = df['month'].map(season_map)
    
    return df

train = add_temporal_features(train, DATE_COL)
val = add_temporal_features(val, DATE_COL)
print(f'After temporal features: train={train.shape}')

## 2. Interaction Features

In [ ]:
def add_interaction_features(df):
    """Create domain-informed interaction features."""
    df = df.copy()
    
    # Runoff proxy: precipitation × slope
    if 'precip_sum_7d' in df.columns and 'basin_slope_deg' in df.columns:
        df['runoff_proxy'] = df['precip_sum_7d'] * df['basin_slope_deg']
    
    # Agricultural phosphorus load
    if 'basin_agriculture_pct' in df.columns and 'precip_sum_7d' in df.columns:
        df['agri_phosphorus_load'] = df['basin_agriculture_pct'] * df['precip_sum_7d']
    
    # Dilution factor
    if 'river_discharge_cms' in df.columns and 'basin_upstream_area_km2' in df.columns:
        df['dilution_factor'] = df['river_discharge_cms'] * df['basin_upstream_area_km2']
    
    # Soil weathering index (pH × elevation)
    if 'soil_phh2o' in df.columns and 'elevation_m' in df.columns:
        df['weathering_index'] = df['soil_phh2o'] * df['elevation_m']
    
    # Soil clay-organic interaction (affects nutrient binding)
    if 'soil_clay' in df.columns and 'soil_ocd' in df.columns:
        df['clay_organic_interaction'] = df['soil_clay'] * df['soil_ocd']
    
    return df

train = add_interaction_features(train)
val = add_interaction_features(val)
print(f'After interactions: train={train.shape}')

## 3. Log-Transform Skewed Features

In [ ]:
# Identify highly skewed numeric features (|skew| > 2)
TARGET_COLS = []  # adjust — auto-detect same as notebook 02
for col in train.columns:
    cl = col.lower()
    if any(k in cl for k in ['alkalinity', 'conductance', 'phosphorus']):
        TARGET_COLS.append(col)

numeric_cols = train.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols if c not in TARGET_COLS]

skewed_features = []
for col in feature_cols:
    skew = train[col].skew()
    if abs(skew) > 2 and train[col].min() >= 0:  # only log-transform non-negative
        skewed_features.append((col, skew))

print(f'Features with |skew| > 2: {len(skewed_features)}')
for col, skew in sorted(skewed_features, key=lambda x: abs(x[1]), reverse=True)[:10]:
    print(f'  {col}: skew={skew:.2f}')

# Apply log1p transform
for col, _ in skewed_features:
    train[f'{col}_log'] = np.log1p(train[col])
    val[f'{col}_log'] = np.log1p(val[col]) if col in val.columns else np.nan

print(f'\nAfter log transform: train={train.shape}')

## 4. One-Hot Encode Categoricals

In [ ]:
# Encode season and any other categorical features
cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()
# Remove ID-like columns
cat_cols = [c for c in cat_cols if c not in ['GEMS_Station_Number', 'River_Name', 'Sample_Date']]

print(f'Categorical columns to encode: {cat_cols}')

if cat_cols:
    train = pd.get_dummies(train, columns=cat_cols, drop_first=True)
    val = pd.get_dummies(val, columns=cat_cols, drop_first=True)
    
    # Align columns (val may have missing dummies)
    for col in train.columns:
        if col not in val.columns and col not in TARGET_COLS:
            val[col] = 0
    
print(f'After encoding: train={train.shape}, val={val.shape}')

## 5. Feature Selection (Per-Target)
RFECV with GroupKFold (station-aware) — respects spatial structure.

In [ ]:
STATION_COL = 'GEMS_Station_Number'  # adjust
META_COLS = [STATION_COL, 'Latitude', 'Longitude', 'Sample_Date', 'River_Name',
             'month', 'quarter', 'day_of_year', 'year']  # columns NOT to use as features

all_feature_cols = [c for c in train.select_dtypes(include=[np.number]).columns 
                    if c not in TARGET_COLS + META_COLS]

print(f'Total candidate features: {len(all_feature_cols)}')

# Groups for spatial CV during feature selection
groups = train[STATION_COL]
gkf = GroupKFold(n_splits=5)

In [ ]:
# RFECV per target — this takes a while
selected_features = {}

for target in TARGET_COLS:
    target_key = target.split('_')[0][:12]  # short name
    print(f'\n{"="*50}')
    print(f'Feature Selection for: {target}')
    print(f'{"="*50}')
    
    # Drop rows with null targets
    valid_mask = train[target].notna()
    X_sel = train.loc[valid_mask, all_feature_cols].fillna(0)
    y_sel = train.loc[valid_mask, target]
    groups_sel = groups[valid_mask]
    
    # Use RF as the estimator (fast, handles nonlinear)
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)
    
    rfecv = RFECV(
        estimator=rf,
        step=5,  # remove 5 features per iteration (faster)
        cv=gkf.split(X_sel, y_sel, groups=groups_sel),
        scoring='r2',
        min_features_to_select=10,
        n_jobs=-1
    )
    rfecv.fit(X_sel, y_sel)
    
    selected = [all_feature_cols[i] for i in range(len(all_feature_cols)) if rfecv.support_[i]]
    selected_features[target_key] = selected
    
    print(f'  Selected {len(selected)} / {len(all_feature_cols)} features')
    print(f'  Best CV R²: {rfecv.cv_results_["mean_test_score"].max():.4f}')
    print(f'  Top 10: {selected[:10]}')

## 6. Save Results

In [ ]:
# Save featured datasets
train.to_parquet(f'{WORK_DIR}/train_featured.parquet', index=False)
val.to_parquet(f'{WORK_DIR}/val_featured.parquet', index=False)

# Save feature sets as YAML
feature_config = {}
for key, feats in selected_features.items():
    feature_config[key] = {'features': feats, 'count': len(feats)}

# Common features (intersection)
if len(selected_features) > 1:
    common = set.intersection(*[set(f) for f in selected_features.values()])
    feature_config['common'] = {'features': sorted(list(common)), 'count': len(common)}

with open(f'{WORK_DIR}/feature_sets.yaml', 'w') as f:
    yaml.dump(feature_config, f, default_flow_style=False)

print(f'✅ Saved train_featured.parquet: {train.shape}')
print(f'✅ Saved val_featured.parquet: {val.shape}')
print(f'✅ Saved feature_sets.yaml')
print(f'\nFeature counts per target:')
for key, feats in selected_features.items():
    print(f'  {key}: {len(feats)} features')